# Link-level channel

> Every bit of every transmission, through a link chain: the radio channel of the link level.

In [ ]:
#| default_exp radio.link.channel

In [ ]:
#| hide
from nbdev.showdoc import *

Link-level simulation in the sense of [Sionna PHY](https://nvlabs.github.io/sionna/phy/index.html): what happens
to the bits of each transmission. `LinkLevelChannel` is the radio channel (see `radio.channel`) whose receiver is a
link chain (see `radio.link.chain`): `UncodedChain`, `NRChain` (see `radio.link.presets`), or one of your own, as
`SystemLevelChannel` takes any `ErrorModel`. It adds the `SINR` block when the chain needs it (link adaptation).

It is slower than `SystemLevelChannel`, which skips the bits, and it takes the same fading, power control and
interference: the same scenario runs at either level.

* **Delivery.** Without a CRC (`UncodedChain`), a sent message is delivered with its bit errors (`success=True`):
  the payload the receiver gets may differ from what was sent; `drop_corrupted=True` drops corrupted packets
  instead. With one (`NRChain`), the CRC decides.
* **Outages** are lost, as at the system level.

`size_bits` is what the codec produced. `error_rate` is the transmission's bit error rate, and `metadata` adds
`bit_errors` (and, with `NRChain`, `mcs`, `resource_elements` and `tb_size`).

In [ ]:
#| export
from __future__ import annotations

from comm_core.block import Block
from comm_core.channel.fading import Fading
from comm_core.channel.interference import SINR
from comm_core.radio.channel import RadioChannel
from comm_core.radio.link.chain import LinkChain
from comm_core.radio.link.presets import UncodedChain
from comm_core.radio.power_control import PowerControl

In [ ]:
#| export
class LinkLevelChannel(RadioChannel):
    "Every bit simulated, through a link chain (`UncodedChain`, `NRChain`, or one of your own)."

    def __init__(
        self,
        chain: LinkChain | None = None, # The link chain; None: `UncodedChain()` (BPSK, no code)
        fading: Fading | None = None, # The coefficient h of a transmission; None: h = 1
        power_control: PowerControl | None = None, # Transmit power and outages; None: the link's tx_power_w
        noise_power: float = 1.0, # Noise power at the receiver, per symbol (same unit as the power)
        seed: int | None = None, # Seed of the noise and the fading
        interference: bool = False, # A step's transmissions share the medium (see `RadioChannel`)
        half_duplex: bool = True, # With interference: a node that transmits in a step cannot receive in it
    ):
        self.chain = chain if chain is not None else UncodedChain()
        self.sinr = SINR()
        super().__init__(fading, power_control, noise_power, seed, interference, half_duplex)

    def receiver(self) -> list:
        "The chain, after the SINR if it needs it (link adaptation)."
        return [self.sinr, self.chain] if self.chain.needs_sinr else [self.chain]

In [ ]:
show_doc(LinkLevelChannel.receiver)

---

[source](https://github.com/Ahmed-Khaled-Saleh/comm-core/blob/main/comm_core/radio/link/channel.py#L39){target="_blank" style="float:right; font-size:smaller"}

### LinkLevelChannel.receiver

```python
def receiver()->list:
```

*The chain, after the SINR if it needs it (link adaptation).*

### Tests

In [ ]:
import math

import numpy as np

from comm_core.core import Message, Transmission
from comm_core.net.topology import Link
from comm_core.source.codec import UniformQuantizer, to_bits
from comm_core.radio.system.phy_abstraction import bpsk_bit_error_rate, packet_error_rate
from comm_core.channel.fading import NoFading, RayleighFading
from comm_core.radio.power_control import ChannelInversion, MessagePower
from comm_core.radio.system.channel import SystemLevelChannel

In [ ]:
def send(ch, payload, link=Link('a', 'b'), n=1, **metadata):
    t = Transmission(message=Message('a', 'b', payload, metadata=metadata), sender='a', receiver='b', start_time=0.0)
    return [ch.transmit(t, link) for _ in range(n)]

idx = np.arange(49) * 5                                          # 49 codebook indices
r, = send(LinkLevelChannel(noise_power=1e-6, seed=0), idx)        # 60 dB: no errors
assert r.success and (r.payload == idx).all() and r.size_bits == 49 * 8 and r.metadata['bit_errors'] == 0
assert abs(r.snr_db - 60) < 1e-9 and r.energy == 49 * 8 and r.latency == 0
# 0 dB: about 8% of the bits are wrong, and the receiver gets the corrupted indices
rs = send(LinkLevelChannel(noise_power=1.0, seed=0), idx, n=50)
ber = np.mean([r.error_rate for r in rs])
assert all(r.success for r in rs) and abs(ber - bpsk_bit_error_rate(1.0)) < 0.01
assert not (rs[0].payload == idx).all()
# a CRC: corrupted packets are lost
assert not any(r.success for r in send(LinkLevelChannel(UncodedChain(drop_corrupted=True), noise_power=1.0, seed=0), idx, n=20))
# channel inversion: the weak link (|h|^2 = 0.01) would need 1000 > 10: outage, nothing sent
for h, sent in ((1.2, True), (0.1, False)):
    r, = send(LinkLevelChannel(fading=NoFading(h), power_control=ChannelInversion(10, max_power=10), seed=0), idx)
    assert r.success == sent and r.size_bits == (392 if sent else 0) and r.metadata.get('outage', False) != sent
    if sent:
        assert abs(r.snr_db - 10) < 1e-6
# the sender chooses the power
ch = LinkLevelChannel(power_control=MessagePower(), seed=0)
assert not send(ch, idx)[0].success and send(ch, idx, tx_power=100.0)[0].metadata['tx_power'] == 100.0
# a finite bandwidth: latency = symbols / bandwidth, energy = power x latency
r, = send(LinkLevelChannel(noise_power=1e-6), idx, link=Link('a', 'b', bandwidth_hz=1e6, tx_power_w=0.5))
assert abs(r.latency - 392e-6) < 1e-15 and abs(r.energy - 0.5 * 392e-6) < 1e-15
# Rayleigh fading: varying SNR, reproducible with reset(seed)
ch = LinkLevelChannel(fading=RayleighFading(), noise_power=0.1)
ch.reset(seed=1); first = [r.snr_db for r in send(ch, idx, n=5)]
ch.reset(seed=1); assert [r.snr_db for r in send(ch, idx, n=5)] == first and len(set(first)) == 5
# real numbers: quantized to 8 bits
r, = send(LinkLevelChannel(UncodedChain(UniformQuantizer(8, -1, 1)), noise_power=1e-6), np.array([0.25, -0.5]))
assert r.size_bits == 16 and np.allclose(r.payload, [0.25, -0.5], atol=1 / 255)

In [ ]:
from comm_core.net.backend import PipelineBackend
from comm_core.radio.power_control import FixedPower
from comm_core.radio.channel import Airtime

In [ ]:
# the link level is its blocks: the same results from them in a pipeline, and a batch is one at a time
ch = LinkLevelChannel(fading=RayleighFading(), noise_power=0.5)
blocks = PipelineBackend([RayleighFading(), FixedPower(), UncodedChain(), Airtime()], noise_power=0.5)
assert [type(b).__name__ for b in ch.pipeline()] == [type(b).__name__ for b in blocks.pipeline]
t = Transmission(message=Message('a', 'b', idx), sender='a', receiver='b', start_time=0.0)
ch.reset(seed=2); one = [ch.transmit(t, Link('a', 'b')).error_rate for _ in range(20)]
ch.reset(seed=2); batch = [r.error_rate for r in ch.transmit_batch([(t, Link('a', 'b'))] * 20)]
assert one == batch and len(set(one)) > 1

In [ ]:
from comm_core.modulation.constellation import SionnaModulation
from comm_core.radio.link.presets import NRChain

In [ ]:
#| sionna
# 16-QAM through the link level: a quarter of the symbols, and the errors of its SNR
ch = LinkLevelChannel(UncodedChain(modulation=SionnaModulation('qam', 4)), noise_power=1e-3, seed=0)
r, = send(ch, idx)
assert r.success and (r.payload == idx).all() and r.metadata['bit_errors'] == 0 and r.energy == 392 / 4
rs = send(LinkLevelChannel(UncodedChain(modulation=SionnaModulation('qam', 4)), noise_power=10 ** -1.2, seed=0), idx, n=100)
ber = np.mean([r.error_rate for r in rs])
assert 0.02 < ber < 0.04                                        # 12 dB: about 0.028

The two levels agree. With a CRC, the link-level channel loses packets as often as the system level's PHY abstraction
says it does:

In [ ]:
idx = np.arange(8) * 30                                          # 64 bits
for snr_db in (4, 6, 8):
    noise = 10 ** (-snr_db / 10)
    link_level = LinkLevelChannel(UncodedChain(drop_corrupted=True), noise_power=noise, seed=0)
    lost = 1 - np.mean([r.success for r in send(link_level, idx, n=1000)])
    expected = packet_error_rate(snr_db, 64)
    assert abs(lost - expected) < 0.04, (snr_db, lost, expected)

With interference (`interference=True`, see `radio.channel`), the link level adds it to the noise of every bit, and gives the BER of the SINR:

In [ ]:
from comm_core.net.backend import AnalyticalBackend
from comm_core.channel.ray_tracing import RayTracedFading

In [ ]:
xs = np.arange(5)
d = np.abs(np.subtract.outer(xs, xs)).astype(float)
table = RayTracedFading(np.where(d > 0, 1e-3 / np.maximum(d, 1) ** 1.5, 0), [(x, 0) for x in xs],
                        node_positions={'a': (0, 0), 'b': (1, 0), 'c': (2, 0), 'd': (3, 0)})
backend = AnalyticalBackend(LinkLevelChannel(fading=table, noise_power=1e-10, seed=0, interference=True))
pair = lambda s, r: (Transmission(Message(s, r, np.arange(256) % 256), s, r, 0.0), Link(s, r))
r, _ = backend.transmit_all([pair('a', 'b'), pair('c', 'd')])        # c is as close to b as a: SINR about 0 dB
sinr = 1e-6 / (1e-10 + 1e-6)
assert abs(r.snr_db - 10 * np.log10(sinr)) < 1e-9 and abs(r.metadata['interference_power'] - 1e-6) < 1e-18
assert abs(r.error_rate - bpsk_bit_error_rate(sinr)) < 0.02 and r.success

### 5G NR

The tests of `NRChain` through the channel. They need Sionna, so they are flagged (`#| sionna`): `nbdev_test --flags
sionna` runs them.

**The link level against the system level.** At MCS 10, 1000-bit messages are sent over a flat channel. The
fraction lost (the transport block error rate, TBLER) is compared with what `SionnaErrorModel` predicts. The PHY
abstraction should place the error-rate curve within about a dB of the simulation.

In [ ]:
from comm_core.channel.fading import NoFading
from comm_core.radio.power_control import ChannelInversion
from comm_core.radio.system.phy_abstraction import SionnaErrorModel

In [ ]:
#| sionna
def send_all(channel, payloads, link=Link('a', 'b')):
    pairs = [(Transmission(message=Message('a', 'b', p), sender='a', receiver='b', start_time=0.0), link) for p in payloads]
    return channel.transmit_batch(pairs)

rng = np.random.default_rng(0)
codec = UniformQuantizer(bits=8, low=0, high=1)                        # 125 numbers: 1000 bits
payloads = [rng.uniform(0, 1, 125) for _ in range(200)]
model = SionnaErrorModel(mcs_index=10)
print(' SINR   simulated TBLER, 20 / 50 decoder iterations   SYS predicted')
for snr_db in (2.5, 3.0, 3.5, 4.0, 4.5, 5.0):
    lost = []
    for iterations in (20, 50):
        channel = LinkLevelChannel(NRChain(codec, mcs_index=10, num_bp_iter=iterations), noise_power=10 ** (-snr_db / 10), seed=1)
        lost.append(1 - np.mean([r.success for r in send_all(channel, payloads)]))
    predicted = model([np.array([10 ** (snr_db / 10)])], np.array([1000]))[0][0]
    print(f'{snr_db:4.1f} dB {lost[0]:21.3f} / {lost[1]:.3f} {predicted:20.3f}')
    if predicted <= 0.01:
        assert lost[1] <= 0.2
    if predicted >= 0.99:
        assert lost[1] >= 0.8


 SINR   simulated TBLER, 20 / 50 decoder iterations   SYS predicted


 2.5 dB                 1.000 / 0.990                0.414


 3.0 dB                 0.960 / 0.695                0.120


 3.5 dB                 0.490 / 0.130                0.035


 4.0 dB                 0.040 / 0.000                0.009


 4.5 dB                 0.005 / 0.000                0.001


 5.0 dB                 0.000 / 0.000                0.000


* **With 50 decoder iterations,** the link level agrees with Sionna SYS where its tables have data (around 4 dB here).
* **With 20 iterations** (`TBDecoder`'s default, and so `num_bp_iter`'s), it needs about half a dB more.
* **Between table points,** the system level's curve is an interpolation. Its tables step 1.8 dB in SINR, so its
  waterfall is shallower than the real one.

**What arrives.** Codebook indices (49 of them, 8 bits each), with link adaptation: a delivered message is exactly
what was sent, and a lost one has failed its CRC.

In [ ]:
#| sionna
indices = np.arange(49) * 5
for snr_db in (0, 10, 25):
    channel = LinkLevelChannel(NRChain(), noise_power=10 ** (-snr_db / 10), seed=0)   # IndexCodec(256), link adaptation
    rs = send_all(channel, [indices] * 20)
    ok = [r for r in rs if r.success]
    print(f'{snr_db:2d} dB: MCS {rs[0].metadata["mcs"]:2d}, {rs[0].metadata["resource_elements"]:4d} resource elements, '
          f'{len(ok)}/20 delivered')
    assert all((r.payload == indices).all() and r.metadata['bit_errors'] == 0 for r in ok)
    assert all(r.payload is None for r in rs if not r.success)
    assert rs[0].size_bits == 392
# reproducible with the seed; tiny payloads; an outage
a = [r.success for r in send_all(LinkLevelChannel(NRChain(mcs_index=10), noise_power=10 ** -0.35, seed=3), [indices] * 30)]
b = [r.success for r in send_all(LinkLevelChannel(NRChain(mcs_index=10), noise_power=10 ** -0.35, seed=3), [indices] * 30)]
assert a == b
r, = send_all(LinkLevelChannel(NRChain(mcs_index=10), noise_power=0.01, seed=0), [np.array([7, 200])])   # 16 bits
assert r.success and r.payload.tolist() == [7, 200]
r, = send_all(LinkLevelChannel(NRChain(), fading=NoFading(0.1), power_control=ChannelInversion(10, max_power=10)), [indices])
assert not r.success and r.metadata['outage']

 0 dB: MCS  5,  530 resource elements, 15/20 delivered


10 dB: MCS 17,  153 resource elements, 18/20 delivered


25 dB: MCS 27,   75 resource elements, 20/20 delivered


**Small messages: where Sionna SYS is wrong.** Link adaptation targets 10% losses using Sionna SYS's predictions.
At the link level, messages of a few hundred bits or more meet the target. Small messages are lost far more
often: Sionna's tables were simulated for code blocks from 24 bits, at coarse SINR steps, and are much too
optimistic for small transport blocks at high code rates. For small messages, use a fixed, lower `mcs_index`, or
check the MCS at the link level.

In [ ]:
#| sionna
rng = np.random.default_rng(0)
print('bits  SINR  MCS  REs  lost (target 0.1)')
lost = {}
for numbers in (2, 8, 49, 125):
    for snr_db in (5, 15, 25):
        channel = LinkLevelChannel(NRChain(UniformQuantizer(8, 0, 1), num_bp_iter=50), noise_power=10 ** (-snr_db / 10), seed=0)
        rs = send_all(channel, [rng.uniform(0, 1, numbers) for _ in range(100)])
        lost[8 * numbers, snr_db] = 1 - np.mean([r.success for r in rs])
        print(f'{8 * numbers:4d} {snr_db:5d} {rs[0].metadata["mcs"]:4d} {rs[0].metadata["resource_elements"]:4d}  {lost[8 * numbers, snr_db]:.2f}')
assert all(lost[392, s] <= 0.2 for s in (5, 15, 25))                   # a few hundred bits: about the target

bits  SINR  MCS  REs  lost (target 0.1)


  16     5   10   13  1.00


  16    15   21    8  0.84


  16    25   26   17  1.00


  64     5   11   44  0.59


...

 392    15   22  101  0.04


 392    25   27   75  0.00


1000     5   12  590  0.23


1000    15   22  257  0.00


1000    25   27  188  0.00


**In a network**, over a channel ray-traced at anchors with 52 subcarriers (here line-of-sight paths along a
corridor). Four nodes take turns (`TDMAProtocol`) and share the medium (`interference=True`):

In [ ]:
from comm_core import Network, Node, FullMeshTopology, TDMAProtocol, AnalyticalBackend, RayTracedPathsFading, grid_points

In [ ]:
#| sionna
C, F = 299_792_458.0, 5.18e9
points = grid_points((0, 0), (30, 0), 0.5, height=1.0)
v = points[None] - points[:, None]
L = np.linalg.norm(v, axis=-1)
same = L == 0
L = np.where(same, 1.0, L)
paths = dict(a=np.where(same, 0, C / F / (4 * np.pi * L) * np.exp(-2j * np.pi * F * L / C))[..., None],
             tau=(L / C)[..., None], u_tx=(v / L[..., None])[..., None, :], u_rx=(-v / L[..., None])[..., None, :])
fading = RayTracedPathsFading(points, points, **paths, frequency=F, subcarriers=(np.arange(52) - 25.5) * 312.5e3)
nodes = ['a', 'b', 'c', 'd']
fading.set_positions({'a': (0.3, 0, 1), 'b': (6.1, 0, 1), 'c': (14.8, 0, 1), 'd': (29.6, 0, 1)})
channel = LinkLevelChannel(NRChain(), fading=fading, noise_power=10 ** ((-94 - 30) / 10), seed=0, interference=True)
net = Network([Node(n) for n in nodes], FullMeshTopology(nodes), TDMAProtocol(nodes, queue_size=3), AnalyticalBackend(channel))
for link in (net.topology.get_link(s, r) for s in nodes for r in nodes if s != r):
    link.tx_power_w, link.bandwidth_hz = 1e-4, 20e6                     # 0.1 mW, 20 MHz
indices = np.arange(49) * 5
for step in range(4):
    for s in nodes:
        for r in nodes:
            if s != r:
                net.send(Message(s, r, indices))
    for res in net.step():
        t = res.transmission
        print(f'{t.sender} -> {t.receiver}: SINR {res.snr_db:5.1f} dB, MCS {res.metadata["mcs"]:2d}, '
              f'{res.metadata["resource_elements"]:4d} REs, delivered {res.success}')
received = net.receive('a')
assert net.metrics.summary().transmissions == 12 and all((m.payload == indices).all() for m in received)

a -> b: SINR  22.0 dB, MCS 27,   75 REs, delivered True
a -> c: SINR  14.0 dB, MCS 21,  109 REs, delivered True
a -> d: SINR   7.9 dB, MCS 14,  182 REs, delivered True
b -> a: SINR  22.0 dB, MCS 27,   75 REs, delivered True
b -> c: SINR  18.5 dB, MCS 25,   82 REs, delivered True
b -> d: SINR   9.8 dB, MCS 17,  153 REs, delivered False
c -> a: SINR  14.0 dB, MCS 21,  109 REs, delivered True
c -> b: SINR  18.5 dB, MCS 25,   82 REs, delivered True
c -> d: SINR  13.9 dB, MCS 21,  109 REs, delivered False


d -> a: SINR   7.9 dB, MCS 14,  182 REs, delivered True
d -> b: SINR   9.8 dB, MCS 17,  153 REs, delivered False
d -> c: SINR  13.9 dB, MCS 21,  109 REs, delivered True


A broadcast: one MCS for all its receivers, the weakest one's (see `SionnaErrorModel`).

In [ ]:
#| sionna
net.reset()
for r in ('b', 'c', 'd'):
    net.send(Message('a', r, indices))
unicast = {res.transmission.receiver: res.metadata['mcs'] for res in net.step()}
net.reset()
net.broadcast('a', indices)
results = net.step()
broadcast = {res.transmission.receiver: res.metadata['mcs'] for res in results}
print('unicast MCSs:', unicast, ' broadcast:', broadcast, ' delivered:', [res.success for res in results])
assert len(set(unicast.values())) > 1 and set(broadcast.values()) == {min(unicast.values())}

unicast MCSs: {'b': 27, 'c': 21, 'd': 14}  broadcast: {'b': 14, 'c': 14, 'd': 14}  delivered: [True, True, True]


## Batched, without a network

The pieces also work on their own: training code can apply the same steps to a whole batch of links at once, on numpy arrays or torch tensors, e.g.
`B·T` samples × `n` neighbours, each sending 49 codebook indices, with channel inversion. Here `csi` holds the
coefficients `(B·T, n, 1)`, and links in an outage get the null index `codebook_size`:

```python
codec, bpsk = IndexCodec(256), BPSK()
power, send = ChannelInversion(target_snr_db=10, max_power=10).allocate(csi.abs() ** 2, noise_power)
x = bpsk.modulate(codec.encode(msg_indices[:, None, :]))           # (B·T, 1, 392), the same message to every neighbour
y = apply_channel(x, csi, power, noise_power)                        # (B·T, n, 392)
received = codec.decode(bpsk.demodulate(y, csi))                   # (B·T, n, 49)
received = torch.where(send, received, codec.codebook_size)        # nothing received in an outage
```

In [ ]:
from comm_core.channel.awgn import apply_channel
from comm_core.modulation.constellation import BPSK
from comm_core.source.codec import IndexCodec

In [ ]:
#| hide
def batch(xp, msg_indices, csi, noise_power=1.0, rng=None):
    codec, bpsk = IndexCodec(256), BPSK()
    power, send = ChannelInversion(target_snr_db=10, max_power=10).allocate(abs(csi) ** 2, noise_power)
    x = bpsk.modulate(codec.encode(msg_indices[:, None, :]))
    y = apply_channel(x, csi, power, noise_power, rng)
    received = codec.decode(bpsk.demodulate(y, csi))
    return xp.where(send, received, codec.codebook_size), send

rng = np.random.default_rng(0)
msg = rng.integers(0, 256, (64, 49))
csi = (rng.standard_normal((64, 3, 1)) + 1j * rng.standard_normal((64, 3, 1))) / np.sqrt(2)
received, sent = batch(np, msg, csi, rng=rng)
assert received.shape == (64, 3, 49) and ((received == 256) == ~sent).all()
ber = (to_bits(np.where(sent, received, 0), 8) != to_bits(np.broadcast_to(msg[:, None], received.shape), 8))[sent[..., 0]].mean()
assert abs(ber - bpsk_bit_error_rate(10.0)) < 1e-3                # every sent link at 10 dB
try:
    import torch
except ImportError:
    torch = None
if torch is not None:                                               # the same on torch tensors
    g = torch.Generator().manual_seed(0)
    received_t, sent_t = batch(torch, torch.as_tensor(msg), torch.as_tensor(csi), rng=g)
    assert received_t.shape == (64, 3, 49) and torch.equal(sent_t, torch.as_tensor(sent))
    assert ((received_t == 256) == ~sent_t).all()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()